In [0]:
jdbc_url = ("jdbc:sqlserver://capstone6dbserver.database.windows.net:1433;"
            "database=capstone6db;"
            "user=ltmadmin@capstone6dbserver;"
            "password=P@55word;"
            "encrypt=true;"
            "trustServerCertificate=false;"
            "hostNameInCertificate=*.database.windows.net;"
            "loginTimeout=30;"
)

try:
    region = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.region_id")
        .option("user", "ltmadmin")
        .option("password", "P@55word")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(region)
except Exception as e:
    print(str(e)[:2000])

In [0]:
try:
    tower = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.tower_id")
        .option("user", "ltmadmin")
        .option("password", "P@55word")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(tower)
except Exception as e:
    print(str(e)[:2000])

In [0]:
region.write.mode("overwrite").saveAsTable("capstone6_telecom.telecom.region_id")
tower.write.mode("overwrite").saveAsTable("capstone6_telecom.telecom.tower_id")

In [0]:
cat = "capstone6_telecom.telecom"
ev  = f"{cat}.bronze_network_events"

In [0]:
display(spark.sql(f"""
SELECT 'network_events' AS source, COUNT(*) AS actual, 11540 AS expected FROM {ev}
UNION ALL SELECT 'status_updates',   COUNT(*), 520 FROM {cat}.bronze_status_updates
UNION ALL SELECT 'secondary_events', COUNT(*), 310 FROM {cat}.bronze_secondary_events
UNION ALL SELECT 'region_id',        COUNT(*), 110 FROM {cat}.region_id
UNION ALL SELECT 'tower_id',         COUNT(*), 150 FROM {cat}.tower_id
"""))

In [0]:
display(spark.sql(f"SELECT source_file, COUNT(*) AS rows FROM {ev} GROUP BY source_file ORDER BY source_file"))
display(spark.sql(f"DESCRIBE TABLE {ev}"))

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS total_rows,                                   -- 11,540
       COUNT(DISTINCT measurement_id) AS unique_ids,             -- 11,500
       COUNT(*) - COUNT(DISTINCT measurement_id) AS duplicate_rows  -- 40
FROM {ev}"""))

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS exact_duplicate_groups FROM (
  SELECT measurement_id, event_timestamp, ingest_timestamp, region_id, tower_id,
         network_type, network_status, unit, latency_ms, packet_loss_pct,
         operator_id, source_system
  FROM {ev}
  GROUP BY ALL
  HAVING COUNT(*) > 1
)"""))

In [0]:
display(spark.sql(f"""
SELECT measurement_id, COUNT(*) AS copies, collect_set(source_file) AS files
FROM {ev}
GROUP BY measurement_id
HAVING COUNT(*) > 1
ORDER BY measurement_id"""))

In [0]:
cols = ["measurement_id","event_timestamp","ingest_timestamp","region_id","tower_id",
        "network_type","network_status","unit","latency_ms","packet_loss_pct",
        "operator_id","source_system"]
sel = ", ".join([f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols])
display(spark.sql(f"SELECT {sel} FROM {ev}"))